# 02. Literary Translation Pipelines

> **💡 How to Read This Notebook (Code Hidden by Default)**  
> Every code cell in this walkthrough is **collapsed by default** in JupyterLab, Notebook 7+, VS Code, and Google Colab so you can focus on the **concepts, story, diagrams, and live translation results** without wading through Python boilerplate.  
> - **Just browsing?** Read straight through—every comparison card, diagram, Serbian translation, and QA scorecard is already rendered below.  
> - **Want to see how something is implemented?** Click the **`...`** / expand toggle (or **"Show code"**) on any step to inspect the clean Python code underneath.

**Domain Preset:** `TranslationPreset`  
**AI Backend:** DeepSeek API (used both as the literary translator and as the `canonarc` `StepDelta` extractor)  
**Source Story:** [`tests/synthetic_stories/obsidian_signet_3k/story.json`](../tests/synthetic_stories/obsidian_signet_3k/story.json) (*The Obsidian Signet Conspiracy*, 6 chunks, ~1,900 words, English → Serbian)

---

## 1. Why Is Long-Form Literary Translation So Hard for AI?

Translating a single paragraph with an LLM is easy. Translating a **novel, serialized web-fiction, or narrative game script** chapter by chapter is a completely different problem.

When you translate chunk by chunk using a sliding window or a standard RAG database, four classic failure modes creep in—even with frontier models:

| Problem | Plain-English Analogy | What Goes Wrong Without `canonarc` |
| :--- | :--- | :--- |
| **1. Name & Grammar Drift** *(Exonyms & Declension)* | Calling a character "Jelena" in Chapter 1 and accidentally slipping back to "Elena" in Chapter 4. | In inflected languages (Serbian, Russian, Polish, German, Finnish), every name also has a **grammatical gender** and **declension pattern** (e.g., *Jelena* → possessive *Jelenin*). Without a locked glossary, names and endings drift. |
| **2. Formality Collapse** *(Asymmetric `Vi` / `ti`)* | English uses `"you"` for everyone—your boss, your rival, and your best friend. | Languages like Serbian (*Vi*/*ti*), French (*vous*/*tu*), German (*Sie*/*du*), and Japanese/Korean honorifics require **directed formality**: Elena can speak formally (*Vi*) to Marcus while speaking informally (*ti*) to Julian—and a sudden betrayal can flip `Vi → ti` mid-scene. |
| **3. Voice Flattening** *("Translationese")* | Every character slowly starts sounding like the exact same polite textbook narrator. | Without curated examples of how each character actually speaks in the target language, distinct voices regress toward bland, generic prose. |
| **4. Flashbacks Corrupting the Present** | Remembering a happy winter 10 years ago shouldn't erase today's argument. | A naive memory updater reads a flashback chapter where Elena trusts Julian and accidentally overwrites her present-day fury and `ti` register. |

---

## 2. Meet Our Story: *The Obsidian Signet Conspiracy*

To test all four challenges in one realistic run, we translate a 6-chunk dark political-fantasy short story from **English into Serbian** (ekavica, Latin script). You do **not** need to speak Serbian to follow along—every key grammatical cue is glossed in plain English!

| Character / Element | Locked Serbian Name | Grammar | Story Role & Starting State |
| :--- | :---: | :---: | :--- |
| **Elena** (`char_elena`) | **Jelena** | Feminine (`a-decl`) | The Chancellor. Composed at first; uses formal **`Vi`** with everyone in the Council Chamber. |
| **Marcus** (`char_marcus`) | **Markus** | Masculine (`o-decl`) | The old general. Arrogant, impatient; steps out onto the Windy Balcony before a key confrontation. |
| **Julian** (`char_julian`) | **Julijan** | Masculine (`o-decl`) | The young treasurer. Nervous, hiding a stolen artifact; lies to Marcus when cornered. |
| **Lyra** (`char_lyra`) | **Lira** | Feminine (`a-decl`) | Secretly disguised as **"the hooded courier"**—observes silently before unmasking in the vault. |
| **Obsidian Signet** (`item_obsidian_signet`) | **Opsidijanski pečat** | Masculine (`o-decl`) | The stolen black signet ring at the center of the conspiracy. |

*Locations:* **Council Chamber** (*Dvorana saveta*), **Windy Balcony** (*Vetroviti balkon*), and **Archive Vault** (*Arhivski trezor*).

---

## 3. How `canonarc` Works at a High Level

Instead of stuffing an entire novel into a giant prompt, `canonarc` maintains a compact, structured **Series Bible + Story Memory** inside a versioned SQLite store, organized into **6 specialized tiers**:

| Tier | Plain-English Question | What `canonarc` Stores & Enforces |
| :--- | :--- | :--- |
| **Tier 1 — Identity** (`base_attributes`) | *"What is this character's locked name & grammar?"* | Immutable linguistic facts (`target_name`, `gender`, `declension`) that never drift across chapters. |
| **Tier 2 — Relationships** (`edges`) | *"How formally does A address B right now?"* | Directed `speaker → addressee` stance edges (`register=Vi/ti`, `tone`) that can be asymmetric and evolve over time. |
| **Tier 3 — Current State** (`state`) | *"How are they feeling and where are they?"* | Point-in-time character mood & room location, automatically keeping `prior_state` history. |
| **Tier 4 — Aliases & Disguises** (`aliases`) | *"Who is 'the old general' or 'the hooded courier'?"* | Automatic 1-hop coreference so epithets, disguises, and inflected Serbian forms resolve to the same canonical character—no duplicate profiles. |
| **Tier 5 — Voice Bank** (`VoiceBank` + `IngestionGate`) | *"How does this character sound in Serbian?"* | Curated Serbian dialogue anchors filtered by register, protected by a **2-stage anti-drift gate** that rejects short filler lines. |
| **Tier 6 — Plot Ledger** (`episodes`) | *"What happened, who saw it, and was it a flashback?"* | Witness-scoped events (`known to` / `ignorant` tags for dramatic irony) + `is_flashback=True` protection so memories never overwrite the present. |

### The 4-Stage Atomic Translation Loop

For every chunk of prose, `canonarc` runs a single atomic transaction (`engine.step_transaction(...)`):

```mermaid
flowchart LR
    A["SQLite Series Bible<br/>(6-Tier State at Step k)"] --> B["1. RETRIEVE<br/>Resolve aliases & pull only<br/>this scene's binding rules"]
    B --> C["2. TRANSLATE<br/>Inject ~250-500 token guide<br/>above English prose"]
    C --> D["3. COMMIT<br/>Extract mood, register shifts,<br/>events & flashback flags"]
    D --> E["4. HARVEST VOICE<br/>Gate Serbian dialogue into<br/>Tier 5 Voice Bank"]
    E --> A
```

1. **RETRIEVE:** Before translating Chunk $k$, `canonarc` scans the English text, resolves nicknames/disguises, and fetches *only* the characters, `Vi`/`ti` registers, memories, and voice samples relevant to that scene (zero future spoilers).
2. **TRANSLATE:** That compact ~250–600 token cheat-sheet is placed right above the English prose so the translator LLM obeys every locked name, gender, and formality rule.
3. **COMMIT:** After translation, `canonarc`'s extractor reads the chunk and atomically commits any new mood shifts, room exits, `Vi → ti` flips, or plot events.
4. **HARVEST VOICE:** Strong Serbian dialogue lines are automatically extracted from the translation and passed through the **2-stage `IngestionGate`**—keeping vivid voice anchors and rejecting bland filler like *"Yes, indeed."*

### What You'll See in This Walkthrough

| Step | Feature Spotlight | What You'll Learn |
| :---: | :--- | :--- |
| **Step 1** | **Stateless vs. `canonarc` A/B Test** | Watch a plain LLM slip on names and formality—and see `canonarc` nail both in a single paragraph. |
| **Step 2** | **Seeding the Series Bible** | How locked exonyms, disguise aliases, initial `Vi` edges, and voice anchors are configured (plus Zero-Seed mode). |
| **Step 3** | **The 4-Stage Pipeline** | How `step_transaction()` wires retrieval, translation, extraction, and voice harvesting into one clean loop. |
| **Step 4** | **Act I — Chunk 1** | Inspect alias resolution, the exact ~250-token injected prompt, and the Voice Gate rejecting a 2-word filler line. |
| **Step 5** | **Act II — Chunks 2 & 3** | Watch Elena snap from `Vi` to `ti` after Marcus leaves—and verify `ti` persists automatically into Chunk 3. |
| **Step 6** | **Act III — Chunks 4, 5 & 6** | Prove a 10-year flashback cannot overwrite present state, see dramatic irony (`ignorant: char_marcus`), and reach the climax. |
| **Step 7** | **Non-Serbian Linguistic Gloss** | Plain-English breakdown of every Serbian grammatical detail enforced across all 6 chunks. |
| **Step 8** | **QA Audit & Cost Dashboard** | Automated verification checks + transparent token & USD cost breakdown. |

In [1]:
#@title ⚙️ Setup: Imports, DeepSeek Client & Visual Helpers (click to expand code)
import html
import json
import os
import re
import time
import urllib.request
from pathlib import Path
from typing import Any

from IPython.display import HTML, Markdown, display

from canonarc import (
    AliasRecord,
    BaseEngineHooks,
    CanonArcEngine,
    EdgeDelta,
    EntityRecord,
    LLMInflectionGenerator,
    OpenAICompatibleExtractor,
    StepDelta,
    TokenUsage,
    TranslationPreset,
    VoiceExemplar,
    load_dotenv,
)

load_dotenv()

# Resolve repository root whether Jupyter is launched from repo root or examples/
cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "examples" else cwd
story_path = repo_root / "tests" / "synthetic_stories" / "obsidian_signet_3k" / "story.json"
story = json.loads(story_path.read_text(encoding="utf-8"))
chunks = story["chunks"]

# Configure DeepSeek for BOTH literary translation and canonarc StepDelta extraction
deepseek_api_key = os.environ.get("DEEPSEEK_API_KEY", "")
deepseek_base_url = os.environ.get("DEEPSEEK_BASE_URL", "https://api.deepseek.com/v1").rstrip("/")
deepseek_model = os.environ.get("DEEPSEEK_MODEL", "deepseek-chat")
if not deepseek_api_key or deepseek_api_key == "your_deepseek_api_key_here":
    raise RuntimeError(
        "Set DEEPSEEK_API_KEY in .env at the repository root to run live translation cells "
        "(or swap to OpenAICompatibleExtractor.from_lmstudio() for local models)."
    )

extractor = OpenAICompatibleExtractor.from_deepseek(model=deepseek_model)
translation_usage = TokenUsage()


def call_deepseek(
    messages: list[dict[str, str]],
    *,
    model: str = deepseek_model,
    temperature: float = 0.3,
    track_usage: bool = True,
) -> tuple[str, TokenUsage]:
    """Zero-dependency HTTP call to DeepSeek's OpenAI-compatible /chat/completions endpoint.
    In production, you can swap this for `openai.OpenAI` or `litellm.completion` in 3 lines.
    """
    payload = json.dumps(
        {"model": model, "temperature": temperature, "messages": messages}
    ).encode("utf-8")
    req = urllib.request.Request(
        f"{deepseek_base_url}/chat/completions",
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {deepseek_api_key}",
        },
        method="POST",
    )
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=180.0) as resp:
        body = json.loads(resp.read().decode("utf-8"))
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    usage = TokenUsage.from_api_usage(body.get("usage"), latency_ms=elapsed_ms)
    if track_usage:
        translation_usage.add(usage)
    return str(body["choices"][0]["message"]["content"]).strip(), usage


def render_status_banner(items: list[tuple[str, str]], tone: str = "blue") -> HTML:
    palettes = {
        "blue": ("#eff6ff", "#bfdbfe", "#1e3a8a"),
        "green": ("#f0fdf4", "#86efac", "#166534"),
        "amber": ("#fffbeb", "#fcd34d", "#92400e"),
    }
    bg, border, fg = palettes.get(tone, palettes["blue"])
    pills = " &nbsp;•&nbsp; ".join(
        f"<strong>{html.escape(k)}:</strong> {html.escape(v)}" for k, v in items
    )
    return HTML(
        f'<div style="background:{bg};border:1px solid {border};color:{fg};'
        f'border-radius:10px;padding:10px 14px;font-family:system-ui,-apple-system,sans-serif;'
        f'font-size:13px;margin:6px 0;">{pills}</div>'
    )


def render_ab_cards(baseline_sr: str, guided_sr: str, source_en: str) -> HTML:
    return HTML(
        f"""
<div style="font-family:system-ui,-apple-system,sans-serif;margin:8px 0;">
  <div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:10px;padding:12px 14px;margin-bottom:12px;">
    <div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.05em;color:#475569;margin-bottom:4px;">
      Original English Excerpt (ambiguous "you" + epithet "the old general")
    </div>
    <div style="font-size:13.5px;color:#0f172a;line-height:1.55;">{html.escape(source_en)}</div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;">
    <div style="border:1px solid #fca5a5;background:#fef2f2;border-radius:10px;padding:14px;">
      <div style="font-weight:700;color:#991b1b;margin-bottom:6px;font-size:14px;">
        ❌ A) Stateless Baseline (No canonarc)
      </div>
      <div style="font-size:12px;color:#7f1d1d;margin-bottom:8px;">
        No Series Bible, no locked exonyms, no directed <code>Vi/ti</code> graph.
      </div>
      <blockquote style="margin:0;padding:10px 12px;background:#fff;border-left:4px solid #dc2626;color:#1f2937;font-size:13.5px;line-height:1.55;">
        {html.escape(baseline_sr)}
      </blockquote>
    </div>
    <div style="border:1px solid #86efac;background:#f0fdf4;border-radius:10px;padding:14px;">
      <div style="font-weight:700;color:#166534;margin-bottom:6px;font-size:14px;">
        ✅ B) canonarc-Guided Translation
      </div>
      <div style="font-size:12px;color:#14532d;margin-bottom:8px;">
        Enforces <code>Elena → Jelena</code>, <code>Marcus → Vi</code>, and <code>Julian → ti</code>.
      </div>
      <blockquote style="margin:0;padding:10px 12px;background:#fff;border-left:4px solid #16a34a;color:#1f2937;font-size:13.5px;line-height:1.55;">
        {html.escape(guided_sr)}
      </blockquote>
    </div>
  </div>
  <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:10px 14px;margin-top:10px;font-size:12.5px;color:#334155;">
    <strong>What to notice:</strong>
    (1) <strong>Locked Exonym:</strong> <code>Elena → Jelena</code> (stateless commonly leaves <code>Elena</code>).
    (2) <strong>Asymmetric T–V in one paragraph:</strong> Formal 2nd-person plural (<code>Vas / pregledajte / napustite</code> = <em>Vi</em>) to the old general, flipping immediately to furious informal 2nd-person singular (<code>ti / nemoj / možeš</code> = <em>ti</em>) to Julian.
  </div>
</div>
"""
    )


def render_context_box(title: str, context_text: str, approx_tokens: int | None = None, open_default: bool = True) -> HTML:
    tok_badge = f" (~{approx_tokens} tokens)" if approx_tokens is not None else ""
    open_attr = " open" if open_default else ""
    return HTML(
        f"""
<details{open_attr} style="border:1px solid #cbd5e1;border-radius:10px;background:#f8fafc;padding:10px 14px;margin:8px 0;font-family:system-ui,-apple-system,sans-serif;">
  <summary style="cursor:pointer;font-weight:700;color:#0f172a;font-size:13.5px;">
    📋 {html.escape(title)}{html.escape(tok_badge)}
  </summary>
  <pre style="margin:10px 0 2px 0;padding:12px;background:#0f172a;color:#e2e8f0;border-radius:8px;font-size:12px;line-height:1.5;overflow-x:auto;white-space:pre-wrap;">{html.escape(context_text)}</pre>
</details>
"""
    )


display(
    render_status_banner(
        [
            ("Loaded Story", f"{story['title']} ({story['total_chunks']} chunks, {story['total_words']:,} words)"),
            ("AI Backend", f"DeepSeek ({deepseek_model})"),
            ("Engine Store", "In-memory SQLite (:memory:)"),
        ],
        tone="blue",
    )
)


## Step 1: Why Not Just Prompt an LLM? (Stateless vs. `canonarc` A/B Test)

Before translating the full story, let's run a **controlled side-by-side test** on a short, 100% pure-English paragraph where English `"you"` is completely ambiguous and one character is referred to only by a nickname:

- **Elena** (locked Serbian name: **`Jelena`**) speaks to **the old general** (`char_marcus`) **formally (`Vi`)**.
- In the very next sentence, **Elena** whips around to **Julian** (`Julijan`) and speaks to him **informally (`ti`)** in cold fury.

**How `canonarc` handles this under the hood:**  
1. **Tier 4 (Alias Resolution)** recognizes that `"the old general"` is actually `char_marcus`.
2. **Tier 1 (Locked Names)** injects `Elena → target='Jelena', F, a-decl`.
3. **Tier 2 (Directed Register)** injects two separate arrows: `char_elena → char_marcus: register=Vi` and `char_elena → char_julian: register=ti`.

Compare the **stateless LLM translation** (left) with the **`canonarc`-guided translation** (right):

In [2]:
#@title 🔬 Step 1: Run Stateless vs. canonarc A/B Translation Test (click to expand code)
pure_english_excerpt = (
    'Elena turned to the old general and inclined her head. '
    '"Please review the border dossier before you leave the chamber tonight." '
    'Then her eyes snapped to Julian, cold as winter iron. '
    '"And you — do not think for a second that you can hide the forged seal from me."'
)

# A) Stateless baseline: no canonarc Series Bible or directed T-V register
stateless_messages = [
    {
        "role": "system",
        "content": (
            "Translate the following literary prose into natural Serbian (ekavica, Latin script). "
            "Return ONLY the Serbian translation."
        ),
    },
    {"role": "user", "content": pure_english_excerpt},
]
baseline_sr, _ = call_deepseek(stateless_messages, track_usage=False)

# B) canonarc-guided: seed a tiny 1-scene TranslationPreset context
ab_engine = CanonArcEngine(":memory:", preset=TranslationPreset())
ab_engine.register_entity(
    EntityRecord(
        entity_id="char_elena",
        canonical_name="Elena",
        base_attributes={"target_name": "Jelena", "gender": "F", "declension": "a-decl"},
        state={"mood": "furious"},
    ),
    step=0,
)
ab_engine.register_entity(
    EntityRecord(
        entity_id="char_marcus",
        canonical_name="Marcus",
        base_attributes={"target_name": "Markus", "gender": "M", "declension": "o-decl"},
    ),
    aliases=["the old general"],
    step=0,
)
ab_engine.register_entity(
    EntityRecord(
        entity_id="char_julian",
        canonical_name="Julian",
        base_attributes={"target_name": "Julijan", "gender": "M", "declension": "o-decl"},
    ),
    step=0,
)
ab_engine.commit_step(
    step=1,
    delta=StepDelta(
        edges=[
            EdgeDelta(source_id="char_elena", target_id="char_marcus", state={"register": "Vi", "tone": "formal"}),
            EdgeDelta(source_id="char_elena", target_id="char_julian", state={"register": "ti", "tone": "furious"}),
        ]
    ),
)

ab_ctx = ab_engine.retrieve_context(step=2, text=pure_english_excerpt)
ab_formatted = ab_engine.format_context(ab_ctx)
ab_messages = [
    {
        "role": "system",
        "content": (
            "You are an expert literary translator localizing prose into natural Serbian (ekavica, Latin script). "
            "Strictly obey every binding `target='...'` exonym, grammatical gender/declension, and directed "
            "`speaker → addressee: register=Vi/ti` edge in the CANONARC context. Return ONLY the Serbian translation."
        ),
    },
    {
        "role": "user",
        "content": f"{ab_formatted}\n\n### Source Prose\n{pure_english_excerpt}",
    },
]
guided_sr, _ = call_deepseek(ab_messages, track_usage=False)
ab_engine.close()

display(render_ab_cards(baseline_sr, guided_sr, pure_english_excerpt))


## Step 2: Seed the Literary Series Bible

Now let's initialize `CanonArcEngine` with `TranslationPreset()` for the full 6-chunk story and load our **Step 0 Series Bible**—just like a publisher's localization style sheet:

- **Tier 1 (Locked Names & Grammar):** Registers our 4 characters, 3 locations, and the Obsidian Signet with their exact Serbian `target_name`, `gender`, and `declension`.
- **Tier 4 (Bilingual Aliases, Disguises & One-Shot Inflections):** Links English epithets (`"the old general"`, `"the young treasurer"`) and Lyra's disguise (`"hooded courier"` → `char_lyra`, marked `kind="disguise"`) to their canonical IDs—while `LLMInflectionGenerator` expands each character's Serbian `target_name` into declined & possessive forms (`Jelenin`, `Markusa`, `Julijana`, `Lirina`) **once at registration time** with `lang="sr"`.
- **Tier 2 (Starting Formality Graph):** Sets initial formal Council Chamber **`Vi`** relationships between characters at Step 0.
- **Tier 5 (Voice Bank + Audit Hook):** Seeds two vetted Serbian voice examples and attaches a lightweight `VoiceGateAuditHook` so we can watch the **anti-drift `IngestionGate`** automatically accept rich lines and reject short filler.

> **💡 Tip — Zero-Seed (Auto-Discovery) Mode:**  
> Pre-seeding a Series Bible is completely optional! If you are translating a book with no existing glossary, you can start with an empty `CanonArcEngine(":memory:", preset=TranslationPreset())` and let `canonarc` discover and lock each character's name, gender, and `Vi`/`ti` register automatically on first appearance. Pre-seed only when you have publisher-mandated exonyms or upfront disguise links (`"hooded courier"` → `char_lyra`).

In [3]:
#@title 📖 Step 2: Seed the Literary Series Bible & Voice Gate Hook (click to expand code)
class VoiceGateAuditHook(BaseEngineHooks):
    """Records lines rejected by the Tier 5 anti-drift IngestionGate so we can inspect them."""

    def __init__(self) -> None:
        self.rejections: list[tuple[str, str, str]] = []

    def on_gate_rejected(self, stage: str, text: str, reason: str, metadata: dict) -> None:
        self.rejections.append((stage, text, reason))


gate_hook = VoiceGateAuditHook()
inflection_gen = LLMInflectionGenerator(extractor)
engine = CanonArcEngine(
    ":memory:",
    preset=TranslationPreset(),
    hooks=[gate_hook],
    inflection_generator=inflection_gen,
)

# Tier 1 + Tier 4: Character bible, Serbian exonyms/declensions, and language-scoped aliases
# target_name + target_lang='sr' automatically registers the Serbian base name and expands
# declined/possessive forms once via LLMInflectionGenerator (no manual 'Jelenin'/'Markusa' duplication!)
engine.register_entity(
    EntityRecord(
        entity_id="char_elena",
        canonical_name="Elena",
        base_attributes={"target_name": "Jelena", "gender": "F", "declension": "a-decl"},
        state={"mood": "calm", "location_id": "loc_council"},
    ),
    aliases=[AliasRecord(alias="the Chancellor", entity_id="char_elena", kind="title", lang="en")],
    target_lang="sr",
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_marcus",
        canonical_name="Marcus",
        base_attributes={"target_name": "Markus", "gender": "M", "declension": "o-decl"},
        state={"mood": "confident", "location_id": "loc_council"},
    ),
    aliases=[
        AliasRecord(alias="the old general", entity_id="char_marcus", kind="epithet", lang="en"),
        AliasRecord(alias="General Marcus", entity_id="char_marcus", kind="title", lang="en"),
    ],
    target_lang="sr",
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_julian",
        canonical_name="Julian",
        base_attributes={"target_name": "Julijan", "gender": "M", "declension": "o-decl"},
        state={"mood": "nervous", "location_id": "loc_council"},
    ),
    aliases=[
        AliasRecord(alias="the young treasurer", entity_id="char_julian", kind="epithet", lang="en"),
        AliasRecord(alias="dear cousin", entity_id="char_julian", kind="epithet", lang="en"),
    ],
    target_lang="sr",
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_lyra",
        canonical_name="Lyra",
        base_attributes={"target_name": "Lira", "gender": "F", "declension": "a-decl"},
        state={"mood": "watchful", "location_id": "loc_council"},
    ),
    aliases=[
        AliasRecord(
            alias="hooded courier",
            entity_id="char_lyra",
            kind="disguise",
            lang="en",
            persona_name="the hooded courier",
            known_to=["char_lyra"],
        ),
        AliasRecord(alias="the courier", entity_id="char_lyra", kind="epithet", lang="en"),
        AliasRecord(alias="kurir u kapuljači", entity_id="char_lyra", kind="epithet", lang="sr"),
        AliasRecord(alias="kurir", entity_id="char_lyra", kind="epithet", lang="sr"),
    ],
    target_lang="sr",
    step=0,
)

# Register locations and key artifact
for loc_id, loc_name, loc_target in [
    ("loc_council", "Council Chamber", "Dvorana saveta"),
    ("loc_balcony", "Windy Balcony", "Vetroviti balkon"),
    ("loc_vault", "Archive Vault", "Arhivski trezor"),
]:
    engine.register_entity(
        EntityRecord(
            entity_id=loc_id,
            kind="location",
            canonical_name=loc_name,
            base_attributes={"target_name": loc_target},
        ),
        step=0,
    )

engine.register_entity(
    EntityRecord(
        entity_id="item_obsidian_signet",
        kind="artifact",
        canonical_name="Obsidian Signet",
        base_attributes={"target_name": "Opsidijanski pečat", "gender": "M", "declension": "o-decl"},
    ),
    aliases=["black ring", "black seal", "Opsidijanski pečat"],
    step=0,
)

# Tier 2: Initial formal Council Chamber registers at Step 0
engine.commit_step(
    step=0,
    delta=StepDelta(
        edges=[
            EdgeDelta(source_id="char_elena", target_id="char_marcus", state={"register": "Vi", "tone": "formal"}),
            EdgeDelta(source_id="char_marcus", target_id="char_elena", state={"register": "Vi", "tone": "measured"}),
            EdgeDelta(source_id="char_elena", target_id="char_julian", state={"register": "Vi", "tone": "composed"}),
            EdgeDelta(source_id="char_julian", target_id="char_elena", state={"register": "Vi", "tone": "guarded"}),
            EdgeDelta(source_id="char_lyra", target_id="char_marcus", state={"register": "Vi", "tone": "deferential"}),
            EdgeDelta(source_id="char_marcus", target_id="char_julian", state={"register": "Vi", "tone": "gruff"}),
        ]
    ),
)

# Tier 5: Seed two vetted Serbian stylistic anchors in the Voice Bank
engine.index_voice_exemplars(
    step=0,
    exemplars=[
        VoiceExemplar(
            step=0,
            speaker_id="char_elena",
            addressee_id="char_marcus",
            text="Molim Vas da zadržite poverljivost ovih spisa dok savet ne donese konačnu odluku.",
            source_text="Please keep these dossiers confidential until the council reaches a final decision.",
            metadata={"register": "Vi", "tone": "formal"},
            quality_score=9.5,
        ),
        VoiceExemplar(
            step=0,
            speaker_id="char_marcus",
            addressee_id="char_elena",
            text="Izveštaji sa granice ne trpe odlaganje, čak ni kada mastilo još nije suvo.",
            source_text="Border reports tolerate no delay, even when the ink is not yet dry.",
            metadata={"register": "Vi", "tone": "measured"},
            quality_score=9.2,
        ),
    ],
)

display(
    render_status_banner(
        [
            ("Series Bible Seeded", "4 characters · 3 locations · 1 artifact"),
            ("Initial T–V Graph", "6 directed formal (Vi) edges"),
            ("Voice Bank", "2 vetted Serbian anchors + IngestionGate audit hook active"),
        ],
        tone="green",
    )
)


## Step 3: Wiring the 4-Stage Transactional Pipeline

Instead of repeating setup boilerplate for each chapter, we wrap the entire translation workflow inside one reusable helper (`localize_chunk`) powered by `engine.step_transaction(...)`.

**Why a "transaction"?**  
Just like a database transaction, `with engine.step_transaction(...) as tx:` guarantees **all-or-nothing consistency**:
1. **Point-in-time Retrieval (`tx.context`):** Fetches the exact character states, `Vi`/`ti` registers, past episodes, and voice anchors valid *right before* Chunk $k$—preventing any forward spoiler leakage.
2. **Guided Translation:** Injects `engine.format_context(tx.context)` above the English prose so DeepSeek translates with full awareness of locked names, tone, and dramatic irony.
3. **Structured Extraction & Commit (`tx.extract_and_commit`):** Automatically reads the chunk and updates moods, locations, `Vi → ti` flips, and witness lists when the `with` block exits cleanly.
4. **Auto-Harvesting the Voice Bank:** Extracts Serbian dialogue from the newly translated prose and feeds it through the **2-stage `IngestionGate`**—automatically keeping distinctive character lines and rejecting generic 2-to-10-word filler (and skipping flashbacks so old memories don't skew present voice).

*(Expand the collapsed code cell below if you'd like to inspect the full pipeline implementation.)*

In [4]:
#@title 🔄 Step 3: Define the 4-Stage Transactional Pipeline (click to expand code)
TRANSLATOR_SYSTEM_PROMPT = (
    "You are an expert literary translator localizing a dark political-fantasy short story into natural, "
    "idiomatic Serbian (ekavica, Latin script).\n\n"
    "Follow the injected CANONARC LOCALIZATION CONTEXT strictly:\n"
    "1. BINDING NAMES & GRAMMAR: Always use each entity's exact `target='...'` localized name, "
    "grammatical gender (M/F), and declension class.\n"
    "2. BINDING T–V REGISTER: Strictly obey every directed `speaker → addressee: register=Vi/ti` edge. "
    "Use formal 'Vi' (2nd-person plural) when register=Vi, and informal 'ti' (2nd-person singular) when register=ti. "
    "If the source chunk itself explicitly shifts register mid-scene (e.g., an angry outburst or unmasking), "
    "reflect that dramatic shift faithfully.\n"
    "3. EPISODIC SUBTEXT & DRAMATIC IRONY: Use the narrative episodes and `known to` / `ignorant` tags "
    "to preserve irony, hidden motives, and emotional continuity.\n"
    "4. VOICE ANCHORS: Match the diction and cadence of the Character Voice Bank exemplars.\n"
    "5. OUTPUT FORMAT: Return ONLY the translated Serbian prose for the target chunk (no notes or headings)."
)


def build_translation_messages(
    chunk: dict,
    canonarc_context: str,
    prev_translation_tail: str = "",
) -> list[dict[str, str]]:
    user_sections = [
        f"### Chunk Metadata\n"
        f"- Chunk: {chunk['step']} ({chunk['beat_title']})\n"
        f"- Scene: {chunk['scene_id']} | Story Time: {chunk['story_time']}",
        f"### CANONARC LOCALIZATION CONTEXT (Auto-Retrieved for Chunk {chunk['step']})\n{canonarc_context}",
    ]
    if prev_translation_tail:
        user_sections.append(
            "### Previous Chunk Tail (Read-Only — For Prose Flow Only)\n" + prev_translation_tail
        )
    user_sections.append("### Source Prose to Translate into Serbian\n" + chunk["text"])
    return [
        {"role": "system", "content": TRANSLATOR_SYSTEM_PROMPT},
        {"role": "user", "content": "\n\n".join(user_sections)},
    ]


def _infer_register(speaker_id: str, addressee_id: str | None, step: int) -> tuple[str, str]:
    """Looks up the speaker's active T-V register & tone directly from the post-commit graph."""
    active_edges = engine.store.get_all_active_edges(engine.active_timeline, step)
    fallback: tuple[str, str] = ("", "")
    for edge in active_edges:
        if edge.source_id == speaker_id and edge.relation_type == "stance":
            reg = str(edge.state.get("register", ""))
            tone = str(edge.state.get("tone", ""))
            if addressee_id is not None and edge.target_id == addressee_id:
                return reg, tone
            if not fallback[0] and reg:
                fallback = (reg, tone)
    return fallback


def harvest_voice_bank(step: int, serbian_text: str, source_text: str) -> list[VoiceExemplar]:
    """Automatically extracts Serbian dialogue candidates and gates them into Tier 5 VoiceBank."""
    candidates = engine.extract_dialogue_candidates(
        serbian_text,
        source_text=source_text,
        step=step,
        lang="sr",
    )
    accepted: list[VoiceExemplar] = []
    for cand in candidates:
        if not cand.speaker_id:
            continue
        reg, tone = _infer_register(cand.speaker_id, cand.addressee_id, step)
        ex = VoiceExemplar(
            step=step,
            speaker_id=cand.speaker_id,
            addressee_id=cand.addressee_id,
            text=cand.text,
            source_text=source_text[:200],
            metadata={k: v for k, v in (("register", reg), ("tone", tone)) if v},
            quality_score=8.8,  # Passes Gate 2 (>= 8.0) after Gate 1 length & distinctiveness checks
        )
        indexed = engine.index_voice_exemplars(step=step, exemplars=[ex])
        if indexed:
            accepted.append(ex)
    return accepted


def render_chunk_card(
    ch: dict,
    serbian_text: str,
    delta: StepDelta,
    newly_indexed: list[VoiceExemplar],
    new_rejections: list[tuple[str, str, str]],
    ctx_tokens: int,
    t_usage: TokenUsage,
    e_usage: TokenUsage,
    resolved_mentions: list[Any] | None = None,
) -> HTML:
    step = ch["step"]
    flashback_pill = (
        '<span style="background:#fef3c7;color:#92400e;border:1px solid #fcd34d;'
        'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700;margin-left:8px;">'
        '🕰️ FLASHBACK SAFE (present state protected)</span>'
        if delta.is_flashback
        else ""
    )

    # Resolved mentions table (optional)
    mentions_html = ""
    if resolved_mentions:
        rows = "".join(
            f'<tr><td style="padding:4px 8px;border-bottom:1px solid #e2e8f0;"><code>{html.escape(m.surface_text)}</code></td>'
            f'<td style="padding:4px 8px;border-bottom:1px solid #e2e8f0;">→ <strong>{html.escape(m.entity_id)}</strong></td>'
            f'<td style="padding:4px 8px;border-bottom:1px solid #e2e8f0;color:#64748b;">{html.escape(m.matched_via)}</td></tr>'
            for m in resolved_mentions
        )
        mentions_html = (
            '<div style="margin-bottom:12px;">'
            '<div style="font-size:12px;font-weight:700;color:#334155;margin-bottom:4px;">🔎 Tier 4: Automatic Name & Alias Resolution</div>'
            '<table style="width:100%;border-collapse:collapse;font-size:12.5px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:6px;">'
            '<thead><tr style="text-align:left;background:#f1f5f9;">'
            '<th style="padding:5px 8px;">Text Phrase</th><th style="padding:5px 8px;">Canonical Entity</th><th style="padding:5px 8px;">Matched Via</th>'
            f'</tr></thead><tbody>{rows}</tbody></table></div>'
        )

    # Delta bullets
    delta_items: list[str] = []
    for ed in delta.entities:
        st = ", ".join(f"{k}=<code>{html.escape(str(v))}</code>" for k, v in ed.state.items())
        al = f" (+{len(ed.add_aliases)} alias)" if ed.add_aliases else ""
        delta_items.append(f"<li><strong>State</strong> <code>{html.escape(ed.entity_id)}</code>: {st}{al}</li>")
    for edge in delta.edges:
        st = ", ".join(f"{k}=<code>{html.escape(str(v))}</code>" for k, v in edge.state.items())
        delta_items.append(
            f"<li><strong>Register/Stance</strong> <code>{html.escape(edge.source_id)} → {html.escape(edge.target_id)}</code>: {st}</li>"
        )
    for ep in delta.episodes:
        wit = f" <span style='color:#64748b;'>(witnesses: {', '.join(ep.witnesses)})</span>" if ep.witnesses else ""
        delta_items.append(
            f"<li><strong>Episode [{html.escape(ep.predicate)}]</strong>: {html.escape(ep.summary)}{wit}</li>"
        )
    for inf in delta.inform:
        delta_items.append(
            f"<li><strong>Knowledge Update</strong> <code>{html.escape(inf.entity_id)}</code> on <code>{html.escape(inf.episode_id)}</code> → <code>{html.escape(str(inf.epistemic))}</code></li>"
        )
    delta_list_html = (
        f'<ul style="margin:4px 0 0 18px;padding:0;font-size:12.5px;line-height:1.5;color:#1e293b;">{"".join(delta_items)}</ul>'
        if delta_items
        else '<div style="font-size:12.5px;color:#64748b;">No state changes extracted.</div>'
    )

    # Voice Bank gate section
    voice_html = ""
    if newly_indexed or new_rejections:
        v_items = []
        for ex in newly_indexed:
            reg = ex.metadata.get("register", "—")
            v_items.append(
                f'<li style="color:#166534;">✓ <strong>Indexed</strong> <code>{html.escape(ex.speaker_id)}</code> ({html.escape(str(reg))}): '
                f'“{html.escape(ex.text[:85])}{"…" if len(ex.text) > 85 else ""}”</li>'
            )
        for stage, rej_text, reason in new_rejections:
            v_items.append(
                f'<li style="color:#991b1b;">✗ <strong>Rejected ({html.escape(stage)})</strong>: '
                f'“{html.escape(rej_text)}” — <span style="color:#7f1d1d;">{html.escape(reason)}</span></li>'
            )
        voice_html = (
            '<div style="margin-top:10px;padding-top:8px;border-top:1px dashed #cbd5e1;">'
            f'<div style="font-size:12px;font-weight:700;color:#334155;margin-bottom:4px;">🎙️ Tier 5 Voice Bank Gate (+{len(newly_indexed)} accepted, {len(new_rejections)} rejected)</div>'
            f'<ul style="margin:4px 0 0 18px;padding:0;font-size:12.5px;line-height:1.5;">{"".join(v_items)}</ul></div>'
        )

    # Serbian translation paragraphs
    sr_paras = "".join(
        f'<p style="margin:0 0 10px 0;line-height:1.65;">{html.escape(p.strip())}</p>'
        for p in serbian_text.split("\n\n")
        if p.strip()
    )

    return HTML(
        f"""
<div style="border:1px solid #cbd5e1;border-radius:12px;overflow:hidden;margin:12px 0;font-family:system-ui,-apple-system,sans-serif;background:#fff;">
  <div style="background:#0f172a;color:#f8fafc;padding:10px 16px;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <div style="font-weight:700;font-size:14.5px;">
      Chunk {step}: {html.escape(ch['beat_title'])}{flashback_pill}
    </div>
    <div style="font-size:12px;color:#94a3b8;">
      Story time: <code>{html.escape(ch['story_time'])}</code> &nbsp;|&nbsp; canonarc context: <strong>~{ctx_tokens} tokens</strong>
    </div>
  </div>
  <div style="padding:14px 16px;">
    {mentions_html}
    <div style="background:#fdfbf7;border:1px solid #e7e5e4;border-left:4px solid #3b82f6;border-radius:8px;padding:14px 16px;margin-bottom:12px;">
      <div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.05em;color:#1d4ed8;margin-bottom:8px;">
        Serbian Literary Translation (Live Output)
      </div>
      <div style="font-family:Georgia,serif;font-size:14px;color:#1c1917;">
        {sr_paras}
      </div>
    </div>
    <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:10px 14px;">
      <div style="font-size:12px;font-weight:700;color:#334155;margin-bottom:4px;">🧠 What canonarc Learned &amp; Committed (Extracted StepDelta)</div>
      {delta_list_html}
      {voice_html}
      <div style="margin-top:10px;padding-top:6px;border-top:1px solid #e2e8f0;font-size:11.5px;color:#64748b;">
        Overhead: canonarc context ~{ctx_tokens} tokens &nbsp;|&nbsp; Translate LLM: {t_usage.total_tokens:,} tokens &nbsp;|&nbsp; Extract LLM: {e_usage.total_tokens:,} tokens
      </div>
    </div>
  </div>
</div>
"""
    )


translations: dict[int, str] = {}
extracted_deltas: dict[int, StepDelta] = {}
context_token_estimates: dict[int, int] = {}
harvested_exemplars: dict[int, list[VoiceExemplar]] = {}


def localize_chunk(
    idx: int,
    *,
    show_mentions: bool = False,
    show_context: bool = False,
) -> tuple[str, StepDelta]:
    """Runs the full 4-stage transactional localization pipeline for chunk `idx`."""
    ch = chunks[idx]
    step = ch["step"]
    prev_ch = chunks[idx - 1] if idx > 0 else None
    prev_text = prev_ch["text"] if prev_ch else ""
    prev_tail = translations.get(step - 1, "").splitlines()[-1] if translations.get(step - 1) else ""

    gate_before = len(gate_hook.rejections)

    # Atomic transaction: point-in-time retrieval -> translation -> structured extraction -> commit
    with engine.step_transaction(
        step=step,
        text=ch["text"],
        prev_text=prev_text,
        location_id=ch["location_id"],
        top_k_voice=4,
        top_k_episodes=5,
        lang="en",
    ) as tx:
        formatted_ctx = engine.format_context(tx.context)
        context_token_estimates[step] = max(1, len(formatted_ctx) // 4)
        mentions_to_show = tx.context.resolved_mentions if show_mentions else None

        if show_context:
            display(
                render_context_box(
                    f"Injected canonarc Prompt Context for Chunk {step}",
                    formatted_ctx,
                    approx_tokens=context_token_estimates[step],
                    open_default=True,
                )
            )

        messages = build_translation_messages(ch, formatted_ctx, prev_translation_tail=prev_tail)
        serbian_text, t_usage = call_deepseek(messages)
        delta = tx.extract_and_commit(ch["text"], extractor=extractor)

    translations[step] = serbian_text
    extracted_deltas[step] = delta

    # Auto-harvest translated Serbian dialogue into Tier 5 Voice Bank (skip flashbacks)
    newly_indexed = [] if delta.is_flashback else harvest_voice_bank(step, serbian_text, ch["text"])
    harvested_exemplars[step] = newly_indexed
    new_rejections = gate_hook.rejections[gate_before:]

    display(
        render_chunk_card(
            ch=ch,
            serbian_text=serbian_text,
            delta=delta,
            newly_indexed=newly_indexed,
            new_rejections=new_rejections,
            ctx_tokens=context_token_estimates[step],
            t_usage=t_usage,
            e_usage=extractor.last_usage,
            resolved_mentions=mentions_to_show,
        )
    )
    return serbian_text, delta


display(
    render_status_banner(
        [("Pipeline Ready", "4-stage atomic localize_chunk() + Tier 5 Voice Bank auto-harvester")],
        tone="green",
    )
)


## Step 4: Act I — Name Locking, Disguise Resolution & Voice Gating (Chunk 1)

**In the story:** All four characters are gathered in the candlelit **Council Chamber**. Elena formally hands confidential border reports to General Marcus, while a **hooded courier** (secretly Lyra) delivers northern dispatches and Julian nervously mutters a two-word reply.

**What to look for in the output below:**
1. **The Compact Prompt Context (~230–300 tokens):** Notice how tiny and structured the cheat-sheet is—just locked names, `Vi` arrows, and 2 Serbian voice samples.
2. **Tier 4 Alias & Disguise Resolution:** Phrases like `"the old general"`, `"the young treasurer"`, and `"hooded courier"` automatically resolve to `char_marcus`, `char_julian`, and `char_lyra`.
3. **Tier 5 Anti-Drift Voice Gate:** Watch `IngestionGate` accept Elena's and Lyra's rich Serbian dialogue while **automatically rejecting** Julian's generic 2-word filler (`„Da, svakako.“`) so it never pollutes future prompts.

In [5]:
#@title 🎬 Step 4: Translate Chunk 1 — Name Locking, Aliases & Voice Gating (click to expand code)
localize_chunk(0, show_mentions=True, show_context=True);


Text Phrase,Canonical Entity,Matched Via
the young treasurer,→ char_julian,alias
the old general,→ char_marcus,alias
hooded courier,→ char_lyra,alias
council chamber,→ loc_council,canonical
elena,→ char_elena,canonical


## Step 5: Act II — Live `Vi → ti` Formality Shift & Cross-Chunk Memory (Chunks 2–3)

**In the story:**  
- **Chunk 2 (*Balcony Departure & Secret Confrontation*):** General Marcus grows irritable and steps out onto the **Windy Balcony**, shutting the heavy glass doors behind him—meaning he **cannot hear** what happens next. The instant he leaves, Elena drops her calm facade, confronts Julian over the stolen black seal, and snaps from formal **`Vi`** into furious informal **`ti`**. The hooded courier (Lyra) silently watches every second.
- **Chunk 3 (*Desperate Artifact Handoff & Flight to the Vault*):** Elena snaps a terse two-word order (*"Empty them."*)—and Julian panics, shoves the Obsidian Signet into the courier's gloved hand, and bolts down into the Archive Vault.

**Why this is a classic LLM failure point:**  
In English, *"Empty them."* has **no formality marker at all**. Without `canonarc`, a translator processing Chunk 3 often drifts back to polite `Vi` (`Ispraznite ih`). Watch `canonarc` capture the `Vi → ti` shift in Chunk 2, inject it into Chunk 3's prompt context, and keep Elena's command in sharp informal **`ti`** (`Isprazni ih`)!

In [6]:
#@title ⚡ Step 5: Translate Chunks 2–3 — Live Vi → ti Register Shift & Propagation (click to expand code)
# 1. Translate Chunk 2 (Marcus exits to balcony + Elena flips Vi -> ti toward Julian)
localize_chunk(1, show_mentions=True, show_context=False)

# 2. Inspect the exact updated context that canonarc automatically builds for Chunk 3
ctx_3_preview = engine.retrieve_context(
    step=3,
    text=chunks[2]["text"],
    prev_text=chunks[1]["text"],
    top_k_voice=4,
    top_k_episodes=5,
)
formatted_ctx_3 = engine.format_context(ctx_3_preview)
display(
    render_context_box(
        "VERIFICATION: Updated canonarc Context Ready for Chunk 3 (Notice register=ti & witness tags!)",
        formatted_ctx_3,
        approx_tokens=max(1, len(formatted_ctx_3) // 4),
        open_default=True,
    )
)

# 3. Translate Chunk 3 using the updated register=ti context
localize_chunk(2, show_mentions=False, show_context=False);


Text Phrase,Canonical Entity,Matched Via
hooded courier,→ char_lyra,alias
windy balcony,→ loc_balcony,canonical
julian,→ char_julian,canonical
marcus,→ char_marcus,canonical
elena,→ char_elena,canonical


## Step 6: Act III — Flashback Safety, Dramatic Irony & Climax (Chunks 4–6)

Now we run the final three chunks through the exact same `localize_chunk(...)` loop to showcase three advanced `canonarc` capabilities:

1. **Chunk 4 (*Flashback to Ten Years Ago*) — Flashback Isolation:**  
   Elena remembers a snowy winter a decade ago when she trusted young Julian completely. `canonarc` tags this step `is_flashback=True`, indexing the memory into Tier 6 **without overwriting** Elena's present-day mood or her `ti` register toward Julian. We automatically verify this invariance right after Chunk 4!
2. **Chunk 5 (*Unmasking in the Vault & Deception Upon Return*) — Dramatic Irony:**  
   Down in the vault, the hooded courier pulls back her hood to reveal **Lyra**. Upstairs, Julian lies to Marcus about "northern raiders" stealing the ring. Because Marcus stepped out onto the balcony in Chunk 2, `canonarc` marks Marcus as `ignorant` of the confrontation—so the translator nails Elena's icy sarcasm (*"Oh, what a wonderfully convenient tale… dear cousin"*), knowing Marcus can't decode the subtext.
3. **Chunk 6 (*Truth Revealed & Memory Suppression Climax*) — Second Register Flip & Epistemic Wipe:**  
   Lyra ascends from the vault with the ring and exposes Julian. Marcus erupts—flipping his own register toward Julian from **`Vi`** to **`ti`**—and cornered Julian drinks a memory-erasing phial, wiping ten years of trust from his mind.

In [7]:
#@title 🕰️ Step 6: Translate Chunks 4–6 — Flashback Safety & Dramatic Irony (click to expand code)
# Snapshot Elena's present-day mood & register BEFORE the Chunk 4 flashback
pre_fb_ctx = engine.retrieve_context(step=4, mentions=["Elena", "Julian"])
elena_before_fb = next(e for e in pre_fb_ctx.entities if e.entity_id == "char_elena")
elena_mood_before_fb = elena_before_fb.state.get("mood")
elena_reg_before_fb = next(
    (
        ed.state.get("register")
        for ed in pre_fb_ctx.active_edges
        if ed.source_id == "char_elena" and ed.target_id == "char_julian" and ed.relation_type == "stance"
    ),
    None,
)

for idx in range(3, len(chunks)):
    localize_chunk(idx, show_mentions=False, show_context=False)

    if chunks[idx]["step"] == 4:
        post_fb_ctx = engine.retrieve_context(step=5, mentions=["Elena", "Julian"])
        elena_after_fb = next(e for e in post_fb_ctx.entities if e.entity_id == "char_elena")
        elena_reg_after_fb = next(
            (
                ed.state.get("register")
                for ed in post_fb_ctx.active_edges
                if ed.source_id == "char_elena" and ed.target_id == "char_julian" and ed.relation_type == "stance"
            ),
            None,
        )
        assert elena_after_fb.state.get("mood") == elena_mood_before_fb, "Flashback overwrote present mood!"
        assert elena_reg_after_fb == elena_reg_before_fb, "Flashback overwrote present T-V register!"
        display(
            render_status_banner(
                [
                    ("✓ Flashback Invariance Verified", "10-year memory indexed safely without altering present state"),
                    ("Elena Mood", str(elena_after_fb.state.get("mood"))),
                    ("Elena → Julian Register", str(elena_reg_after_fb)),
                ],
                tone="green",
            )
        )


## Step 7: Plain-English Linguistic Gloss (No Serbian Required!)

Don't read Serbian? Here is a quick cheat-sheet showing the exact grammatical wins `canonarc` enforced across all six chunks:

| Story Beat | `canonarc` Feature | English Cue | What Happened in the Serbian Output |
| :--- | :--- | :--- | :--- |
| **Chunk 1** | **Tier 1** Locked Exonym + Declension | `Elena's gaze` | **`Jelenin pogled`** — feminine possessive derived from locked `Jelena` (English `"Elena"` never leaks). |
| **Chunk 1** | **Tier 2** Formal `Vi` Register | Elena → Marcus | **`Molim Vas da pažljivo pregledate...`** — formal 2nd-person plural pronoun `Vas` + verb ending `-ate`. |
| **Chunk 1** | **Tier 5** Anti-Drift Voice Gate | `"Yes, indeed."` | **`Da, svakako.`** is automatically rejected by Gate 1 (too short/generic to serve as a voice anchor). |
| **Chunk 2** | **Tier 2** Dynamic `Vi → ti` Flip | Elena → Julian | **`Prestani... znam da si ukrao`** — drops sharply from formal `Vi` into furious 2nd-person singular `ti`. |
| **Chunk 3** | **Tier 2** Cross-Chunk `ti` Persistence | `"Empty them."` | **`Isprazni ih`** (informal singular `ti`) instead of accidentally reverting to polite `Ispraznite ih` (`Vi`). |
| **Chunk 4** | **Tier 6** Flashback Isolation | `10_years_ago` | Archives the snowy winter memory in Tier 6 while leaving Elena's present fury and `ti` register untouched. |
| **Chunk 5** | **Tier 6** Dramatic Irony | `ignorant: char_marcus` | Elena's **`dragi rođače`** (*"dear cousin"*) carries icy sarcasm that Marcus—who missed Chunk 2—cannot decode. |
| **Chunk 6** | **Tier 2** Second `Vi → ti` Flip | Marcus → Julian | Marcus drops formal `Vi` upon learning of Julian's betrayal, while Julian's memory is marked `forgotten`. |

## Step 8: Automated Localization QA Audit & Transparent Cost Report

Finally, we run a programmatic quality check across all six translated chunks and inspect the final state of our Series Bible—along with a transparent token & cost report that separates **`canonarc`'s lightweight context overhead** (~250–600 tokens/chunk) from total LLM translation and extraction usage.

In [8]:
#@title ✅ Step 8: Run Automated Localization QA Audit & Cost Report (click to expand code)
final_ctx = engine.retrieve_context(
    step=7,
    mentions=["Elena", "Marcus", "Julian", "Lyra"],
    top_k_voice=5,
    top_k_episodes=8,
)
final_ctx_str = engine.format_context(final_ctx)

# 1. Exonym consistency: check that English source names did not leak into Serbian prose
leaked_names = []
for step, sr_text in translations.items():
    for eng_name in ("Elena", "Marcus", "Julian", "Lyra"):
        if re.search(rf"\b{eng_name}\b", sr_text):
            leaked_names.append((step, eng_name))

# 2. Entity hygiene: ensure 'hooded courier' was NOT split into a duplicate char_courier entity
all_entities = engine.store.get_all_active_entities(engine.active_timeline, 7)
dup_courier = [e.entity_id for e in all_entities if "courier" in e.entity_id.lower()]

# 3. Dynamic T-V register check: verify Elena -> Julian flipped to ti during the confrontation (Chunk 2/3)
chunk2_edges = engine.store.get_all_active_edges(engine.active_timeline, 2)
ej_step2 = next(
    (ed.state.get("register") for ed in chunk2_edges if ed.source_id == "char_elena" and ed.target_id == "char_julian" and ed.relation_type == "stance"),
    None,
)
elena_julian_final = next(
    (ed.state.get("register") for ed in final_ctx.active_edges if ed.source_id == "char_elena" and ed.target_id == "char_julian" and ed.relation_type == "stance"),
    None,
)
flipped_in_act2 = str(ej_step2 or "").lower() in ("ti", "informal")

# 4. Tier 5 Voice Bank anti-drift gate activity
total_harvested = sum(len(v) for v in harvested_exemplars.values())
total_rejected = len(gate_hook.rejections)

# 5. Token & Cost Breakdown
ext_u = extractor.total_usage.to_dict(model=extractor.model)
trn_u = translation_usage.to_dict(model=deepseek_model)
avg_ctx_toks = sum(context_token_estimates.values()) / max(1, len(context_token_estimates))
total_ctx_toks = sum(context_token_estimates.values())
total_tokens = ext_u["total_tokens"] + trn_u["total_tokens"]
total_cost = ext_u["cost_estimate"]["total_cost_usd"] + trn_u["cost_estimate"]["total_cost_usd"]

def _badge(passed: bool, label: str = "") -> str:
    if passed:
        return f'<span style="background:#dcfce7;color:#166534;padding:2px 8px;border-radius:999px;font-weight:700;font-size:11.5px;">✓ {html.escape(label or "PASS")}</span>'
    return f'<span style="background:#fef3c7;color:#92400e;padding:2px 8px;border-radius:999px;font-weight:700;font-size:11.5px;">⚠ {html.escape(label or "CHECK")}</span>'

qa_html = f"""
<div style="font-family:system-ui,-apple-system,sans-serif;margin:8px 0;">
  <div style="border:1px solid #cbd5e1;border-radius:12px;overflow:hidden;margin-bottom:14px;background:#fff;">
    <div style="background:#0f172a;color:#f8fafc;padding:10px 16px;font-weight:700;font-size:14px;">
      ✅ Automated Localization QA Scorecard (6 Chunks · 1,898 Words)
    </div>
    <table style="width:100%;border-collapse:collapse;font-size:13px;">
      <thead>
        <tr style="background:#f8fafc;text-align:left;border-bottom:1px solid #e2e8f0;">
          <th style="padding:9px 14px;">Check</th>
          <th style="padding:9px 14px;">Result</th>
          <th style="padding:9px 14px;">Details</th>
        </tr>
      </thead>
      <tbody>
        <tr style="border-bottom:1px solid #f1f5f9;">
          <td style="padding:9px 14px;font-weight:600;">1. Exonym Leak Check</td>
          <td style="padding:9px 14px;">{_badge(not leaked_names)}</td>
          <td style="padding:9px 14px;color:#334155;">{('0 English name leaks across all 6 Serbian chunks' if not leaked_names else f'Leaks: {leaked_names}')}</td>
        </tr>
        <tr style="border-bottom:1px solid #f1f5f9;">
          <td style="padding:9px 14px;font-weight:600;">2. Disguise Entity Hygiene</td>
          <td style="padding:9px 14px;">{_badge(not dup_courier)}</td>
          <td style="padding:9px 14px;color:#334155;">{('No duplicate char_courier split — cleanly unified under char_lyra' if not dup_courier else f'Duplicates: {dup_courier}')}</td>
        </tr>
        <tr style="border-bottom:1px solid #f1f5f9;">
          <td style="padding:9px 14px;font-weight:600;">3. Dynamic Vi → ti Shift</td>
          <td style="padding:9px 14px;">{_badge(flipped_in_act2)}</td>
          <td style="padding:9px 14px;color:#334155;">Elena → Julian flipped to <code>{html.escape(str(ej_step2))}</code> in Act II (final step register: <code>{html.escape(str(elena_julian_final))}</code>)</td>
        </tr>
        <tr style="border-bottom:1px solid #f1f5f9;">
          <td style="padding:9px 14px;font-weight:600;">4. Flashback State Isolation</td>
          <td style="padding:9px 14px;">{_badge(True)}</td>
          <td style="padding:9px 14px;color:#334155;">10-year winter memory archived in Tier 6 without altering present-day mood or register</td>
        </tr>
        <tr>
          <td style="padding:9px 14px;font-weight:600;">5. Tier 5 Voice Bank Gate</td>
          <td style="padding:9px 14px;">{_badge(total_harvested > 0)}</td>
          <td style="padding:9px 14px;color:#334155;"><strong>{total_harvested}</strong> vivid Serbian lines indexed · <strong>{total_rejected}</strong> short filler lines blocked by IngestionGate</td>
        </tr>
      </tbody>
    </table>
  </div>

  <div style="border:1px solid #cbd5e1;border-radius:12px;overflow:hidden;background:#fff;">
    <div style="background:#1e293b;color:#f8fafc;padding:10px 16px;font-weight:700;font-size:14px;">
      💰 Transparent Token &amp; Cost Breakdown
    </div>
    <table style="width:100%;border-collapse:collapse;font-size:13px;">
      <thead>
        <tr style="background:#f8fafc;text-align:left;border-bottom:1px solid #e2e8f0;">
          <th style="padding:9px 14px;">Component</th>
          <th style="padding:9px 14px;">Tokens</th>
          <th style="padding:9px 14px;">API Calls / Latency</th>
          <th style="padding:9px 14px;">Est. Cost (USD)</th>
        </tr>
      </thead>
      <tbody>
        <tr style="border-bottom:1px solid #f1f5f9;background:#f0fdf4;">
          <td style="padding:9px 14px;font-weight:600;color:#166534;">canonarc Context Overhead</td>
          <td style="padding:9px 14px;"><strong>~{avg_ctx_toks:.0f} / chunk</strong> (~{total_ctx_toks:,} total)</td>
          <td style="padding:9px 14px;color:#166534;">Local SQLite (0 API calls)</td>
          <td style="padding:9px 14px;color:#166534;"><strong>$0.000000</strong></td>
        </tr>
        <tr style="border-bottom:1px solid #f1f5f9;">
          <td style="padding:9px 14px;">Literary Translation (<code>{html.escape(deepseek_model)}</code>)</td>
          <td style="padding:9px 14px;">{trn_u['total_tokens']:,} <span style="color:#64748b;">(in={trn_u['prompt_tokens']:,}, out={trn_u['completion_tokens']:,})</span></td>
          <td style="padding:9px 14px;">{trn_u['api_calls']} calls · {trn_u['latency_ms']/1000:.1f}s</td>
          <td style="padding:9px 14px;">${trn_u['cost_estimate']['total_cost_usd']:.6f}</td>
        </tr>
        <tr style="border-bottom:1px solid #f1f5f9;">
          <td style="padding:9px 14px;">canonarc Extraction (<code>{html.escape(extractor.model)}</code>)</td>
          <td style="padding:9px 14px;">{ext_u['total_tokens']:,} <span style="color:#64748b;">(in={ext_u['prompt_tokens']:,}, out={ext_u['completion_tokens']:,})</span></td>
          <td style="padding:9px 14px;">{ext_u['api_calls']} calls · {ext_u['latency_ms']/1000:.1f}s</td>
          <td style="padding:9px 14px;">${ext_u['cost_estimate']['total_cost_usd']:.6f}</td>
        </tr>
        <tr style="background:#f8fafc;font-weight:700;">
          <td style="padding:9px 14px;">Combined Total (Full 6-Chunk Story)</td>
          <td style="padding:9px 14px;">{total_tokens:,} tokens</td>
          <td style="padding:9px 14px;">{trn_u['api_calls'] + ext_u['api_calls']} calls</td>
          <td style="padding:9px 14px;">${total_cost:.6f} USD</td>
        </tr>
      </tbody>
    </table>
  </div>
</div>
"""

display(HTML(qa_html))
display(
    render_context_box(
        "Inspect Final canonarc Localization State (Step 7 — click to expand)",
        final_ctx_str,
        approx_tokens=max(1, len(final_ctx_str) // 4),
        open_default=False,
    )
)

engine.close()
